# Chapter 13: Hopf bifurcations — Brian2

Representative simulations adapted from [`python/chapter13.ipynb`](../python/chapter13.ipynb) and its [chapter notes](../python/chapter13.md).

Coverage: actual supercritical and saturated subcritical Hopf trajectories, with analytical cycle radii overlaid. The separate radial-vector-field and full bifurcation-branch panels remain in the reference. Two initial radii at I=-0.2 show the subcritical system's coexisting basins.

The radial laws are $\dot r=Ir-r^3$ and $\dot r=Ir+r^3-r^5$, with $\dot\theta=1$. Brian2 integrates their nonsingular Cartesian forms.

Run `uv sync --group dev` in the repository root and select that environment's Python kernel. These examples require the source checkout containing `mnd/brian`; an older PyPI installation does not contain these new helpers. Run all cells in order. Brian2 performs every trajectory integration. Numerical states use the book's units; helper clocks map a source time unit to one Brian millisecond. FHN and normal-form time is abstract model time.

In [ ]:
from pathlib import Path
import sys

# Use this source checkout so newly added helpers are available.
root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "mnd").is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))
import brian2 as b2
import numpy as np
import matplotlib.pyplot as plt
b2.prefs.codegen.target = "numpy"

In [ ]:
from mnd.brian.dynamics import simulate_hopf
fig,axes=plt.subplots(1,3,figsize=(12,4))
cases=[("supercritical",-.02,[.2]),("supercritical",.5,[.2,1.2]),("subcritical",-.2,[.2,.8])]
for ax,(kind,current,radii) in zip(axes,cases):
    for radius in radii:
        out=simulate_hopf(kind=kind,current=current,initial=(radius,0),duration=40,dt=.02)
        ax.plot(out.x[0],out.y[0],label=f"r(0)={radius}")
    if current>0 or kind=="subcritical":
        angles=np.linspace(0,2*np.pi,300)
        cycles=[np.sqrt(current)] if kind=="supercritical" else [np.sqrt(.5-np.sqrt(.25+current)),np.sqrt(.5+np.sqrt(.25+current))]
        for k,radius in enumerate(cycles):
            ax.plot(radius*np.cos(angles),radius*np.sin(angles),"k--" if len(cycles)>1 and k==0 else "k:")
    ax.set(xlabel="x",ylabel="y",title=f"{kind}, I={current}",aspect="equal");ax.legend()
plt.tight_layout();plt.show()
assert np.hypot(out.x[0,-1],out.y[0,-1])>.8